# ML4T Chapter 2 Assignment - Complete Solution

This notebook answers all five exercises in **The Financial Data Universe**. It is self-contained except for the public Quandl WIKI Prices data used by Exercises 1, 2, and 4. Set `WIKI_DATA_PATH` to the extracted CSV or Parquet file before rerunning it.

The analysis follows the stated objectives. Conclusions are generated from the displayed results rather than from hard-coded performance claims.


In [1]:
# Shared setup for all five exercises.
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "assignment_support.py").exists():
    NOTEBOOK_DIR = Path("ML4T_Chapter02_Solution").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
print("Notebook support module located successfully.")


Notebook support module located successfully.


## Exercise 1: US Equities EDA

Source notebook: `02_financial_data_universe/01_us_equities_eda.ipynb`. Approximate preparation time: 11 seconds.


In [2]:
from assignment_support import check_ohlc_invariants, load_wiki_prices

wiki = load_wiki_prices()
raw_wiki = wiki
print(f"WIKI panel: {wiki.height:,} rows, {wiki['symbol'].n_unique():,} symbols")
print(f"Date range: {wiki['timestamp'].min()} to {wiki['timestamp'].max()}")


WIKI panel: 3,023,501 rows, 3,199 symbols
Date range: 1980-12-12 to 2018-03-27


### Assignment

The source notebook loads the Quandl WIKI US equities panel and examines coverage, missing exit records, and data quality. Repeat the single-stock checks for another long-lived ticker such as MSFT or GE.

#### Objectives

1. **OHLC invariants:** Filter the full history of the selected stock. Use `check_ohlc_invariants` to test the six invariants on adjusted prices and report each pass rate.
2. **Extreme returns:** Calculate raw-close and adjusted-close returns. List the three largest adjusted losses and gains, then identify whether they coincide with known market events.
3. **Split-date comparison:** Find dates where `split_ratio != 1.0`. Compare raw and adjusted returns and explain why an unadjusted price series is unsuitable for return-based backtests.

#### Guidance

- Use the loaded `wiki` panel directly.
- Daily adjusted return is `adj_close / adj_close.shift(1) - 1`.
- With no economic price move, a split ratio `r` produces a raw return near `1/r - 1`.

#### Expected output

- A six-row invariant report.
- A table of the six most extreme adjusted-return dates with event notes.
- A split-date table showing the mechanical raw-price move and the economically meaningful adjusted return.


In [3]:
# Exercise 1 - Single-stock data-quality review
import numpy as np
import pandas as pd
import polars as pl
# Jupyter provides display() for rich table output.

TARGET_SYMBOL = "MSFT"

stock = wiki.filter(pl.col("symbol") == TARGET_SYMBOL).sort("timestamp")
if stock.is_empty():
    raise ValueError(f"{TARGET_SYMBOL} is not present in the WIKI panel.")

print(
    f"{TARGET_SYMBOL}: {stock.height:,} sessions, "
    f"from {stock['timestamp'].min()} to {stock['timestamp'].max()}"
)

# 1. Six OHLC and volume invariants
invariant_report = check_ohlc_invariants(
    stock,
    open_col="adj_open",
    high_col="adj_high",
    low_col="adj_low",
    close_col="adj_close",
    volume_col="adj_volume",
)
display(invariant_report)

all_invariants_pass = bool((invariant_report["valid_pct"] >= 99.99).all())
print(f"All checks pass at least 99.99% of rows: {all_invariants_pass}")

# 2. Raw and adjusted returns
stock_returns = stock.with_columns(
    (pl.col("close") / pl.col("close").shift(1) - 1).alias("raw_return"),
    (pl.col("adj_close") / pl.col("adj_close").shift(1) - 1).alias("adjusted_return"),
).drop_nulls(["raw_return", "adjusted_return"])

largest_losses = (
    stock_returns.sort("adjusted_return")
    .head(3)
    .with_columns(pl.lit("Largest loss").alias("direction"))
)
largest_gains = (
    stock_returns.sort("adjusted_return", descending=True)
    .head(3)
    .with_columns(pl.lit("Largest gain").alias("direction"))
)

extreme_days = pl.concat([largest_losses, largest_gains]).select(
    "direction",
    "timestamp",
    "close",
    "adj_close",
    "raw_return",
    "adjusted_return",
    "split_ratio",
)

# Add an event note only when the date matches a well-known market event.
# Leave unmatched dates for manual news review rather than guessing a cause.
known_events = {
    "1987-10-19": "Black Monday - broad US equity-market crash",
    "1987-10-20": "Post-Black Monday volatility and rebound",
    "1987-10-21": "Post-Black Monday rebound",
    "2000-04-03": "Antitrust ruling in United States v. Microsoft",
    "2000-04-24": "Microsoft breakup risk during the antitrust case",
    "2008-09-29": "Financial crisis; US House rejected the TARP bill",
    "2008-10-13": "Sharp rebound during the financial crisis",
}

extreme_pd = pd.DataFrame(extreme_days.to_dicts())
extreme_pd["event_note"] = (
    extreme_pd["timestamp"].astype(str).map(known_events).fillna("Manual date-specific news review required")
)
display(extreme_pd)

# 3. Return comparison on split dates
split_days = (
    stock_returns.filter(pl.col("split_ratio").fill_null(1.0) != 1.0)
    .with_columns(
        (1.0 / pl.col("split_ratio") - 1.0).alias("mechanical_split_return"),
        (pl.col("raw_return") - (1.0 / pl.col("split_ratio") - 1.0)).alias(
            "raw_minus_mechanical"
        ),
    )
    .select(
        "timestamp",
        "split_ratio",
        "raw_return",
        "mechanical_split_return",
        "raw_minus_mechanical",
        "adjusted_return",
    )
)
display(split_days)

if split_days.height:
    raw_gap = split_days["raw_minus_mechanical"].abs().mean()
    adjusted_move = split_days["adjusted_return"].abs().mean()
    print(
        f"Mean |raw return - mechanical split return|: {raw_gap:.4%}\n"
        f"Mean |adjusted return| on split dates: {adjusted_move:.4%}"
    )

print(
    "\nConclusion: raw prices contain mechanical jumps caused by stock splits. "
    "Adjusted prices remove these jumps, so backtest returns must be calculated "
    "from adj_close. Using close would create gains or losses that no investor earned."
)


MSFT: 8,076 sessions, from 1986-03-13 to 2018-03-27


check,valid_pct,applicable_rows,total_rows
str,f64,i64,i64
"""high_gte_low""",100.0,8076,8076
"""high_gte_open""",100.0,8076,8076
"""high_gte_close""",100.0,8076,8076
"""low_lte_open""",100.0,8076,8076
"""low_lte_close""",100.0,8076,8076
"""volume_non_negative""",100.0,8076,8076


All checks pass at least 99.99% of rows: True


,direction,timestamp,close,adj_close,raw_return,adjusted_return,split_ratio,event_note
0,Largest loss,1987-10-19,45.25,0.209184,-0.301158,-0.301158,1.0,Black Monday - broad US equity-market crash
1,Largest loss,1987-10-26,39.25,0.181447,-0.186528,-0.186528,1.0,Manual date-specific news review required
2,Largest loss,2000-04-24,66.62,22.174175,-0.156068,-0.156068,1.0,Microsoft breakup risk during the antitrust case
3,Largest gain,2000-10-19,61.88,20.596486,0.195749,0.195749,1.0,Manual date-specific news review required
4,Largest gain,2008-10-13,25.50,20.112357,0.186047,0.186047,1.0,Sharp rebound during the financial crisis
5,Largest gain,1987-10-21,56.63,0.261792,0.179792,0.179792,1.0,Post-Black Monday rebound


timestamp,split_ratio,raw_return,mechanical_split_return,raw_minus_mechanical,adjusted_return
date,f64,f64,f64,f64,f64
1987-09-21,2.0,-0.534783,-0.5,-0.034783,-0.069565
1990-04-16,2.0,-0.496894,-0.5,0.003106,0.006211
1991-06-27,1.5,-0.325062,-0.333333,0.008271,0.012407
1992-06-15,1.5,-0.326667,-0.333333,0.006667,0.01
1994-05-23,2.0,-0.482762,-0.5,0.017238,0.034476
1996-12-09,2.0,-0.465267,-0.5,0.034733,0.069466
1998-02-23,2.0,-0.473861,-0.5,0.026139,0.052279
1999-03-29,2.0,-0.481446,-0.5,0.018554,0.037108
2003-02-18,2.0,-0.48323,-0.5,0.01677,0.03354


Mean |raw return - mechanical split return|: 1.8473%
Mean |adjusted return| on split dates: 3.6117%

Conclusion: raw prices contain mechanical jumps caused by stock splits. Adjusted prices remove these jumps, so backtest returns must be calculated from adj_close. Using close would create gains or losses that no investor earned.


## Exercise 2: Corporate Actions

Source notebook: `02_financial_data_universe/02_corporate_actions.ipynb`. Approximate preparation time: 10 seconds.


In [4]:
print("Corporate-action analysis uses the WIKI panel loaded in Exercise 1.")


Corporate-action analysis uses the WIKI panel loaded in Exercise 1.


### Assignment

The source notebook demonstrates backward adjustment, which anchors the latest observation at a factor of 1. This exercise implements forward adjustment, anchored at the beginning of the series, and separates capital gains from the contribution of reinvested dividends.

#### Objectives

1. **Forward adjustment:** Implement `compute_forward_adjustment(df)`. Corporate-action factors should scale all subsequent prices. Verify that forward- and backward-adjusted daily returns match within floating-point precision.
2. **Total-return decomposition:** Use the final 20 years of AAPL or MSFT. Separate the split-adjusted capital-gain multiple from the total-return multiple and report the dividend contribution as a share of total profit.
3. **Defensive assertions:** Verify that every event factor is positive and that ordinary cash dividends and positive splits produce factors no greater than 1.

#### Expected output

- A successful `np.allclose` comparison and the maximum return difference.
- A 20-year decomposition table.
- Passing assertions for all valid event factors.


In [5]:
# Exercise 2 - Corporate actions and forward adjustment
from datetime import timedelta
import numpy as np
import polars as pl
# Jupyter provides display() for rich table output.

dividend_col = "ex_dividend" if "ex_dividend" in raw_wiki.columns else "ex-dividend"


def compute_forward_adjustment(df: pl.DataFrame) -> pl.DataFrame:
    """Forward-adjust prices with the factor anchored at the first observation.

    The event factor at t is the factor applied to the price at t-1 by backward adjustment:
        ((P[t-1] - dividend[t]) / P[t-1]) / split_ratio[t]
    Forward adjustment scales prices from t onward by the reciprocal of this factor.
    """
    frame = df.sort("timestamp")
    dividend_name = "ex_dividend" if "ex_dividend" in frame.columns else "ex-dividend"

    frame = frame.with_columns(
        pl.col("close").shift(1).alias("previous_close"),
        pl.col("split_ratio").fill_null(1.0).alias("split_ratio_clean"),
        pl.col(dividend_name).fill_null(0.0).alias("dividend_clean"),
    ).with_columns(
        pl.when(pl.col("previous_close").is_null())
        .then(1.0)
        .otherwise(
            (1.0 - pl.col("dividend_clean") / pl.col("previous_close"))
            / pl.col("split_ratio_clean")
        )
        .alias("event_factor")
    )

    invalid = frame.filter(
        pl.col("event_factor").is_null()
        | pl.col("event_factor").is_nan()
        | (pl.col("event_factor") <= 0)
    )
    if invalid.height:
        raise ValueError("A non-positive corporate-action factor was found; inspect the input data.")

    frame = frame.with_columns(
        (1.0 / pl.col("event_factor")).cum_prod().alias("forward_factor")
    )

    price_columns = [c for c in ["open", "high", "low", "close"] if c in frame.columns]
    return frame.with_columns(
        [(pl.col(c) * pl.col("forward_factor")).alias(f"forward_adj_{c}") for c in price_columns]
    )


# 1. Forward and backward adjustment must produce the same daily returns
aapl_raw = raw_wiki.filter(pl.col("symbol") == "AAPL").sort("timestamp")
forward_aapl = compute_forward_adjustment(aapl_raw)

event_factors = forward_aapl["event_factor"].to_numpy()
backward_factors = np.cumprod(event_factors[::-1])[::-1] / event_factors
backward_close = aapl_raw["close"].to_numpy() * backward_factors
forward_returns = forward_aapl["forward_adj_close"].pct_change().to_numpy()
backward_returns = np.r_[np.nan, backward_close[1:] / backward_close[:-1] - 1.0]
valid = np.isfinite(forward_returns) & np.isfinite(backward_returns)

returns_match = np.allclose(
    forward_returns[valid],
    backward_returns[valid],
    rtol=1e-9,
    atol=1e-11,
)
max_return_difference = np.max(np.abs(forward_returns[valid] - backward_returns[valid]))

print(f"Forward and backward returns match: {returns_match}")
print(f"Maximum daily-return difference: {max_return_difference:.3e}")
assert returns_match, "The two adjustment methods do not produce the same daily returns."
vendor_returns = aapl_raw["adj_close"].pct_change().to_numpy()
vendor_gap = np.nanmax(np.abs(backward_returns - vendor_returns))
print(f"Maximum difference from the vendor-adjusted return series: {vendor_gap:.3e}")

# 2. Decompose total performance over the final 20 years
end_date = aapl_raw["timestamp"].max()
start_cutoff = end_date - timedelta(days=round(365.25 * 20))
sample_20y = aapl_raw.filter(pl.col("timestamp") >= start_cutoff).sort("timestamp")

sample_20y = sample_20y.with_columns(
    pl.col("split_ratio").fill_null(1.0).cum_prod().alias("forward_split_factor")
).with_columns(
    (pl.col("close") * pl.col("forward_split_factor")).alias("split_adjusted_close")
)

capital_gain_multiple = (
    sample_20y["split_adjusted_close"][-1] / sample_20y["split_adjusted_close"][0]
)
total_return_multiple = sample_20y["adj_close"][-1] / sample_20y["adj_close"][0]

capital_gain = capital_gain_multiple - 1.0
total_gain = total_return_multiple - 1.0
dividend_reinvestment_gain = total_return_multiple - capital_gain_multiple
dividend_share = 100.0 * dividend_reinvestment_gain / total_gain if total_gain != 0 else np.nan

decomposition = pl.DataFrame(
    {
        "period_start": [sample_20y["timestamp"][0]],
        "period_end": [sample_20y["timestamp"][-1]],
        "capital_gain_multiple": [capital_gain_multiple],
        "total_return_multiple": [total_return_multiple],
        "capital_gain_pct": [100.0 * capital_gain],
        "dividend_reinvestment_gain_pct": [100.0 * dividend_reinvestment_gain],
        "dividend_share_of_total_profit_pct": [dividend_share],
    }
)
display(decomposition)

# 3. Defensive assertions for event factors
factor_audit = forward_aapl.select(
    "timestamp",
    "previous_close",
    "dividend_clean",
    "split_ratio_clean",
    "event_factor",
)

assert factor_audit.filter(pl.col("event_factor") <= 0).is_empty(), (
    "Every adjustment factor must be positive."
)

regular_actions = factor_audit.filter(
    pl.col("previous_close").is_not_null()
    & ((pl.col("dividend_clean") > 0) | (pl.col("split_ratio_clean") > 1.0))
    & (pl.col("dividend_clean") >= 0)
    & (pl.col("dividend_clean") < pl.col("previous_close"))
    & (pl.col("split_ratio_clean") >= 1.0)
)
assert regular_actions.filter(pl.col("event_factor") > 1.0 + 1e-12).is_empty(), (
    "Ordinary dividends and positive splits must have factors no greater than 1."
)

print(
    f"Audited {factor_audit.height:,} dates: every factor is positive; "
    f"all {regular_actions.height:,} ordinary corporate actions have factors <= 1."
)
print(
    "Conclusion: forward and backward adjustment differ only in the price-series anchor. "
    "Returns are identical; the difference between the total-return multiple and the split-only multiple "
    "is the contribution from reinvested dividends."
)


Forward and backward returns match: True
Maximum daily-return difference: 4.298e-16
Maximum difference from the vendor-adjusted return series: 3.348e-04


period_start,period_end,capital_gain_multiple,total_return_multiple,capital_gain_pct,dividend_reinvestment_gain_pct,dividend_share_of_total_profit_pct
date,date,f64,f64,f64,f64,f64
1998-03-27,2018-03-27,174.963623,194.490992,17396.362287,1952.73695,10.092134


Audited 9,400 dates: every factor is positive; all 58 ordinary corporate actions have factors <= 1.
Conclusion: forward and backward adjustment differ only in the price-series anchor. Returns are identical; the difference between the total-return multiple and the split-only multiple is the contribution from reinvested dividends.


## Exercise 3: Point-in-Time Validation

Source notebook: `02_financial_data_universe/14_point_in_time_validation.ipynb`. Approximate preparation time: 6 seconds.


In [6]:
import gc
del wiki, raw_wiki
gc.collect()
print("The point-in-time experiment is fully simulated and reproducible.")


The point-in-time experiment is fully simulated and reproducible.


### Assignment

Fundamental data creates look-ahead bias when the fiscal period end is treated as the date on which the information became public. Build a simulation in which the market reacts only on the filing date, then compare an unavailable look-ahead feature with a point-in-time feature.

#### Objectives

1. **Simulate fundamentals and prices:** Create quarterly surprises for `N_SYMBOLS` stocks. Filing dates must occur `FILING_LAG_MIN` to `FILING_LAG_MAX` days after period end. Daily returns contain noise plus `REACTION * surprise` on the filing date or next trading day.
2. **Build two features:** The look-ahead feature becomes available one day after period end. The point-in-time feature becomes available one day after filing.
3. **Compare:** Calculate mean daily cross-sectional Rank IC and the annualized Sharpe ratio of non-overlapping top-minus-bottom portfolios.

#### Expected output

- One comparison table for both features.
- An explanation of why the look-ahead feature captures the later announcement reaction while the point-in-time feature arrives after that reaction.


In [7]:
# Exercise 3 - Look-ahead bias in simulated data
import numpy as np
import pandas as pd
# Jupyter provides display() for rich table output.

SEED = 42
N_SYMBOLS = 100
START_DATE = "2012-01-01"
END_DATE = "2023-12-31"
FILING_LAG_MIN = 30
FILING_LAG_MAX = 60
REACTION = 0.035
DAILY_VOL = 0.015
FWD_DAYS = 20
TOP_N = 10

rng = np.random.default_rng(SEED)
symbols = [f"STK_{i:03d}" for i in range(N_SYMBOLS)]
trading_days = pd.bdate_range(START_DATE, END_DATE)
quarter_ends = pd.date_range(START_DATE, END_DATE, freq="QE")

# 1. Fundamentals and daily returns
fundamental_rows = []
for symbol in symbols:
    surprises = rng.normal(size=len(quarter_ends))
    lags = rng.integers(FILING_LAG_MIN, FILING_LAG_MAX + 1, size=len(quarter_ends))
    for period_end, lag, surprise in zip(quarter_ends, lags, surprises, strict=True):
        fundamental_rows.append(
            {
                "symbol": symbol,
                "period_end_date": period_end,
                "filing_date": period_end + pd.Timedelta(days=int(lag)),
                "surprise": float(surprise),
            }
        )

fundamentals_sim = pd.DataFrame(fundamental_rows).sort_values(["symbol", "period_end_date"])

returns_matrix = rng.normal(0.0, DAILY_VOL, size=(len(trading_days), N_SYMBOLS))
symbol_position = {symbol: i for i, symbol in enumerate(symbols)}

# The market reacts only on the filing date or the first trading session after it.
for row in fundamentals_sim.itertuples(index=False):
    trading_position = trading_days.searchsorted(row.filing_date)
    if trading_position < len(trading_days):
        returns_matrix[trading_position, symbol_position[row.symbol]] += REACTION * row.surprise

prices_matrix = 100.0 * np.exp(np.cumsum(returns_matrix, axis=0))
daily_sim = pd.DataFrame(
    {
        "date": np.tile(trading_days, N_SYMBOLS),
        "symbol": np.repeat(symbols, len(trading_days)),
        "daily_return": returns_matrix.T.reshape(-1),
        "close": prices_matrix.T.reshape(-1),
    }
)
daily_sim["fwd_return"] = daily_sim.groupby("symbol", sort=False)["close"].transform(
    lambda s: s.shift(-FWD_DAYS) / s - 1.0
)

# 2. Feature availability dates
fundamentals_sim["lookahead_available"] = fundamentals_sim["period_end_date"] + pd.Timedelta(days=1)
fundamentals_sim["pit_available"] = fundamentals_sim["filing_date"] + pd.Timedelta(days=1)


def attach_feature(daily: pd.DataFrame, events: pd.DataFrame, available_col: str) -> pd.DataFrame:
    """Run the as-of join separately by symbol to preserve strict time ordering."""
    pieces = []
    for symbol, left in daily.groupby("symbol", sort=False):
        right = (
            events.loc[events["symbol"] == symbol, [available_col, "surprise"]]
            .sort_values(available_col)
            .rename(columns={available_col: "available_date"})
        )
        merged = pd.merge_asof(
            left.sort_values("date"),
            right,
            left_on="date",
            right_on="available_date",
            direction="backward",
        )
        pieces.append(merged)
    return pd.concat(pieces, ignore_index=True)


lookahead_panel = attach_feature(daily_sim, fundamentals_sim, "lookahead_available")
pit_panel = attach_feature(daily_sim, fundamentals_sim, "pit_available")


def daily_rank_ic(panel: pd.DataFrame) -> pd.Series:
    clean = panel.dropna(subset=["surprise", "fwd_return"])
    return clean.groupby("date").apply(
        lambda g: g["surprise"].rank().corr(g["fwd_return"].rank()),
        include_groups=False,
    ).dropna()


def non_overlapping_long_short(panel: pd.DataFrame) -> pd.Series:
    clean = panel.dropna(subset=["surprise", "fwd_return"])
    rebalance_dates = np.sort(clean["date"].unique())[::FWD_DAYS]
    spreads = []
    for date_value in rebalance_dates:
        cross_section = clean.loc[clean["date"] == date_value].sort_values("surprise")
        if len(cross_section) < 2 * TOP_N:
            continue
        short_return = cross_section.head(TOP_N)["fwd_return"].mean()
        long_return = cross_section.tail(TOP_N)["fwd_return"].mean()
        spreads.append((date_value, long_return - short_return))
    return pd.Series(dict(spreads), name="long_short_return").sort_index()


comparison_rows = []
for label, panel in [("Look-ahead", lookahead_panel), ("Point-in-time", pit_panel)]:
    ic = daily_rank_ic(panel)
    spread = non_overlapping_long_short(panel)
    annualization = np.sqrt(252.0 / FWD_DAYS)
    sharpe = annualization * spread.mean() / spread.std(ddof=1)
    comparison_rows.append(
        {
            "feature": label,
            "mean_daily_rank_ic": ic.mean(),
            "ic_std": ic.std(ddof=1),
            "non_overlap_periods": len(spread),
            "long_short_mean_return": spread.mean(),
            "annualized_sharpe": sharpe,
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({
    "mean_daily_rank_ic": 4,
    "ic_std": 4,
    "long_short_mean_return": 4,
    "annualized_sharpe": 2,
}))

lookahead_ic = comparison.loc[comparison["feature"] == "Look-ahead", "mean_daily_rank_ic"].iat[0]
pit_ic = comparison.loc[comparison["feature"] == "Point-in-time", "mean_daily_rank_ic"].iat[0]
print(
    f"\nLook-ahead IC minus PIT IC: {lookahead_ic - pit_ic:.4f}. "
    "The look-ahead feature knows the surprise before filing and can therefore predict the reaction "
    "embedded in the simulated return. The PIT feature appears only after filing, once the reaction has occurred, "
    "so its predictive power should be substantially lower in this simulation."
)
del daily_sim, lookahead_panel, pit_panel, returns_matrix, prices_matrix
gc.collect()


,feature,mean_daily_rank_ic,ic_std,non_overlap_periods,long_short_mean_return,annualized_sharpe
0,Look-ahead,0.1404,0.1774,153,0.0390,2.78
1,Point-in-time,0.0067,0.1063,151,0.0049,0.52



Look-ahead IC minus PIT IC: 0.1337. The look-ahead feature knows the surprise before filing and can therefore predict the reaction embedded in the simulated return. The PIT feature appears only after filing, once the reaction has occurred, so its predictive power should be substantially lower in this simulation.


## Exercise 4: Survivorship Bias Detection

Source notebook: `02_financial_data_universe/15_survivorship_bias_detection.ipynb`. Approximate preparation time: 24 seconds.


In [8]:
from assignment_support import (
    SCENARIOS, build_panel_matrices as _build_panel_matrices, expected_terminal,
    prepare_survivorship_data, survivor_path, survivorship_chart, universe_path,
)

wiki = load_wiki_prices()
survivorship_data = prepare_survivorship_data(wiki)
globals().update(survivorship_data)

def build_panel_matrices(frame, drop_suspect=True):
    return _build_panel_matrices(frame, window_lifespans, survivors, drop_suspect)

print(f"Analysis window: {window_start} to {window_end}")
print(f"Starting universe: {len(universe):,} symbols; leavers: {leavers.height:,}")


Analysis window: 2014-01-01 to 2018-03-27
Starting universe: 3,095 symbols; leavers: 752


### Assignment

Measure whether survivorship bias differs between high- and low-liquidity stocks during the period in which the WIKI panel records exits.

#### Objectives

1. **Universe segmentation:** For every stock in the starting universe, calculate mean dollar volume during the first month and split the universe at the median.
2. **Portfolios by bucket:** Using repaired returns and `drop_suspect=True`, calculate a survivors-only path and full-universe paths. Evaluate both a zero terminal return and `expected_terminal(SCENARIOS["Empirical 2010-2020"])`.
3. **Compare:** Report symbol counts, exit rates, cumulative returns, and survivorship bias in percentage points. Plot the cumulative paths for both liquidity buckets.

#### Expected output

- A bucket-level comparison table.
- Two cumulative-value charts.
- A conclusion based on the measured differences, including the WIKI panel's coverage limitations when the expected pattern does not appear.


In [9]:
# Exercise 4 - Survivorship bias by liquidity bucket
import numpy as np
import polars as pl
# Jupyter provides display() for rich table output.

# 1. Measure liquidity using only the first month of the analysis window.
initial_liquidity = (
    panel.filter(pl.col("timestamp").dt.date() <= first_month)
    .with_columns((pl.col("close") * pl.col("volume")).alias("dollar_volume"))
    .group_by("symbol")
    .agg(pl.col("dollar_volume").mean().alias("mean_dollar_volume"))
    .filter(pl.col("symbol").is_in(universe))
)

liquidity_median = initial_liquidity["mean_dollar_volume"].median()
bucket_table = initial_liquidity.with_columns(
    pl.when(pl.col("mean_dollar_volume") >= liquidity_median)
    .then(pl.lit("High liquidity"))
    .otherwise(pl.lit("Low liquidity"))
    .alias("bucket")
)

print(f"Median first-month average dollar volume: {liquidity_median:,.0f}")
display(bucket_table.group_by("bucket").agg(pl.len().alias("n_symbols")))

# 2. Calculate survivors-only and full-universe paths for each bucket.
empirical_terminal = expected_terminal(SCENARIOS["Empirical 2010-2020"])
leaver_set = set(leavers["symbol"].to_list())
paths = {}
summary_rows = []

for bucket_name in ["High liquidity", "Low liquidity"]:
    bucket_symbols = bucket_table.filter(pl.col("bucket") == bucket_name)["symbol"].to_list()
    bucket_returns = returns.filter(pl.col("symbol").is_in(bucket_symbols))
    matrices = build_panel_matrices(bucket_returns, drop_suspect=True)

    path_survivors = survivor_path(matrices)
    path_full_zero = universe_path(matrices, terminal=0.0)
    path_full_empirical = universe_path(matrices, terminal=empirical_terminal)

    paths[bucket_name] = {
        "sessions": matrices["sessions"],
        "survivors": path_survivors,
        "full_zero": path_full_zero,
        "full_empirical": path_full_empirical,
    }

    n_symbols = len(bucket_symbols)
    n_leavers = sum(symbol in leaver_set for symbol in bucket_symbols)
    survivor_return = float(path_survivors[-1] - 100.0)
    full_zero_return = float(path_full_zero[-1] - 100.0)
    full_empirical_return = float(path_full_empirical[-1] - 100.0)

    summary_rows.append(
        {
            "bucket": bucket_name,
            "n_symbols": n_symbols,
            "n_leavers": n_leavers,
            "exit_rate_pct": 100.0 * n_leavers / n_symbols,
            "survivor_return_pct": survivor_return,
            "full_return_zero_terminal_pct": full_zero_return,
            "bias_zero_terminal_pp": survivor_return - full_zero_return,
            "full_return_empirical_terminal_pct": full_empirical_return,
            "bias_empirical_terminal_pp": survivor_return - full_empirical_return,
        }
    )

survivorship_summary = pl.DataFrame(summary_rows)
display(survivorship_summary)

# 3. Render both buckets on the same vertical scale for direct comparison.
display(survivorship_chart(paths, empirical_terminal))

summary_pd = pd.DataFrame(survivorship_summary.to_dicts()).set_index("bucket")
low_exit = summary_pd.loc["Low liquidity", "exit_rate_pct"]
high_exit = summary_pd.loc["High liquidity", "exit_rate_pct"]
low_bias = summary_pd.loc["Low liquidity", "bias_empirical_terminal_pp"]
high_bias = summary_pd.loc["High liquidity", "bias_empirical_terminal_pp"]

print(
    f"The low-liquidity bucket has an exit rate of {low_exit:.2f}% versus {high_exit:.2f}% "
    f"for the high-liquidity bucket. Empirical-scenario bias is {low_bias:.2f} and {high_bias:.2f} "
    "percentage points, respectively. The conclusion should follow these measured differences. If the sign is unexpected, "
    "an important limitation is that the WIKI panel records exits only near the end of its history "
    "and does not cover the full market universe."
)


Median first-month average dollar volume: 10,843,068


bucket,n_symbols
str,u32
"""Low liquidity""",1547
"""High liquidity""",1548


bucket,n_symbols,n_leavers,exit_rate_pct,survivor_return_pct,full_return_zero_terminal_pct,bias_zero_terminal_pp,full_return_empirical_terminal_pct,bias_empirical_terminal_pp
str,i64,i64,f64,f64,f64,f64,f64,f64
"""High liquidity""",1548,329,21.25323,38.723174,33.466802,5.256372,33.77103,4.952145
"""Low liquidity""",1547,423,27.343245,44.346613,32.921506,11.425107,33.327442,11.019172


Two-panel survivorship-bias cumulative-return chart

The low-liquidity bucket has an exit rate of 27.34% versus 21.25% for the high-liquidity bucket. Empirical-scenario bias is 11.02 and 4.95 percentage points, respectively. The conclusion should follow these measured differences. If the sign is unexpected, an important limitation is that the WIKI panel records exits only near the end of its history and does not cover the full market universe.


## Exercise 5: File Storage Benchmark

Source notebook: `02_financial_data_universe/20_storage_benchmark_file.ipynb`. Approximate preparation time: 17 seconds.


In [10]:
from assignment_support import (
    estimate_memory_mb, force_materialize_polars, prepare_storage_benchmark,
    time_read, time_write,
)

storage_data = prepare_storage_benchmark()
globals().update(storage_data)
print(f"Synthetic benchmark panel: {ohlcv_df.height:,} rows at scale {ACTIVE_SCALE}")


Synthetic benchmark panel: 1,000,000 rows at scale L


### Assignment

Benchmark Parquet compression and predicate pushdown on the source notebook's one-minute multi-symbol OHLCV panel.

#### Objectives

1. **Compression codecs:** Write the same panel with Snappy, ZSTD, and GZIP. Compare write time, warm-cache read time, and file size.
2. **Predicate pushdown:** Query one symbol over the final 20 percent of the sample. Compare a full `read_parquet` followed by filtering with a lazy `scan_parquet` query that applies row filtering and column projection before collection.
3. **Decision matrix:** Recommend a format and codec for frequently read features and for cold historical archives.

#### Expected output

- A codec benchmark table.
- A full-read versus pushdown table with equal result counts and measured speedup.
- A two-row storage decision matrix based on the observed measurements.


In [11]:
# Exercise 5 - Parquet compression and predicate pushdown
import gc
import polars as pl
# Jupyter provides display() for rich table output.

codecs = ["snappy", "zstd", "gzip"]
codec_rows = []
codec_paths = {}

# 1. Use the same data, row-group size, and timing policy for every codec.
for codec in codecs:
    file_path = BENCHMARK_DIR / f"assignment_ohlcv_{ACTIVE_SCALE.lower()}_{codec}.parquet"
    codec_paths[codec] = file_path

    write_seconds, _ = time_write(
        lambda p=file_path, c=codec: ohlcv_df.write_parquet(
            p,
            compression=c,
            row_group_size=50_000,
            statistics=True,
        )
    )
    file_size_mb = file_path.stat().st_size / 1_000_000

    read_seconds, read_result = time_read(
        lambda p=file_path: force_materialize_polars(pl.read_parquet(p))
    )
    assert read_result.height == ohlcv_df.height

    codec_rows.append(
        {
            "codec": codec,
            "write_seconds": write_seconds,
            "read_seconds": read_seconds,
            "file_size_mb": file_size_mb,
            "rows": read_result.height,
        }
    )

codec_benchmark = pl.DataFrame(codec_rows).sort("file_size_mb")
display(codec_benchmark)

# 2. Run the same query on the ZSTD file using full read and lazy scan.
benchmark_file = codec_paths["zstd"]
target_symbol = ohlcv_df["symbol"].unique().sort()[0]
unique_timestamps = ohlcv_df["timestamp"].unique().sort()
cutoff = unique_timestamps[int(0.80 * (len(unique_timestamps) - 1))]


def full_read_then_filter() -> pl.DataFrame:
    return force_materialize_polars(
        pl.read_parquet(benchmark_file)
        .filter(
            (pl.col("symbol") == target_symbol)
            & (pl.col("timestamp") >= cutoff)
        )
        .select("timestamp", "symbol", "close")
    )


def pushdown_read() -> pl.DataFrame:
    return force_materialize_polars(
        pl.scan_parquet(benchmark_file)
        .filter(
            (pl.col("symbol") == target_symbol)
            & (pl.col("timestamp") >= cutoff)
        )
        .select("timestamp", "symbol", "close")
        .collect()
    )


full_seconds, full_result = time_read(full_read_then_filter)
pushdown_seconds, pushdown_result = time_read(pushdown_read)

assert full_result.height == pushdown_result.height
assert full_result.equals(pushdown_result)

query_benchmark = pl.DataFrame(
    {
        "method": ["Full read, then filter", "Lazy scan + pushdown"],
        "seconds": [full_seconds, pushdown_seconds],
        "result_rows": [full_result.height, pushdown_result.height],
        "result_memory_mb": [
            estimate_memory_mb(full_result),
            estimate_memory_mb(pushdown_result),
        ],
    }
).with_columns(
    (pl.lit(full_seconds) / pl.col("seconds")).alias("speedup_vs_full")
)
display(query_benchmark)

# 3. Base recommendations on measurements from the current machine.
fastest_read_codec = codec_benchmark.sort("read_seconds")["codec"][0]
smallest_codec = codec_benchmark.sort("file_size_mb")["codec"][0]

decision_matrix = pl.DataFrame(
    {
        "use_case": [
            "Frequently read features; prioritize latency",
            "Cold historical archive; prioritize storage",
        ],
        "recommended_format": ["Parquet", "Parquet"],
        "recommended_codec": [fastest_read_codec, smallest_codec],
        "reason": [
            "Use the codec with the lowest measured read time and scan_parquet for row and column pushdown.",
            "Use the codec with the smallest measured file for infrequently accessed historical data.",
        ],
    }
)
display(decision_matrix)

speedup = full_seconds / pushdown_seconds
print(
    f"Pushdown is {speedup:.2f} times faster than full read for this query. "
    "This is a warm-cache benchmark on local storage. With object storage or a network connection, "
    "Parquet's reduction in I/O is often more valuable."
)


codec,write_seconds,read_seconds,file_size_mb,rows
str,f64,f64,f64,i64
"""gzip""",3.970365,0.312611,41.442857,1000000
"""zstd""",1.012387,0.25174,41.509574,1000000
"""snappy""",1.253111,0.230146,48.358242,1000000


method,seconds,result_rows,result_memory_mb,speedup_vs_full
str,f64,i64,f64,f64
"""Full read, then filter""",0.241752,2001,0.043891,1.0
"""Lazy scan + pushdown""",0.010423,2001,0.043891,23.193217


use_case,recommended_format,recommended_codec,reason
str,str,str,str
"""Frequently read features; prio…","""Parquet""","""snappy""","""Use the codec with the lowest …"
"""Cold historical archive; prior…","""Parquet""","""gzip""","""Use the codec with the smalles…"


Pushdown is 23.19 times faster than full read for this query. This is a warm-cache benchmark on local storage. With object storage or a network connection, Parquet's reduction in I/O is often more valuable.
